# MongoDB

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/mongo/mongo.ipynb)

MongoDB is a document database: records are BSON documents in collections, the schema is flexible by default and enforceable with `$jsonSchema`, replica sets provide failover, and sharding provides horizontal scale. Runnable cells use `mongomock`, an in-process stand-in for the `pymongo` API; server-only features (sessions, `explain`, change streams) are shown as fenced snippets.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"mongomock": "mongomock>=4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## What is MongoDB?

The unit of storage is a BSON document: key/value pairs with nested objects, arrays, and rich types (dates, decimals, ObjectId, binary). Documents with a similar shape live in a collection. There is no fixed table schema, joins are optional (`$lookup`, `$graphLookup`), and scale-out is built in through replica sets and shards rather than bolted on.

```text
// mongosh: switch database, insert, query, project
use shop
db.products.insertOne({
  sku: "A1",
  name: "Espresso Machine",
  price: NumberDecimal("249.00"),
  tags: ["kitchen", "coffee"],
  stock: { warehouse: "DFW", qty: 12 }
})
db.products.find({ tags: "coffee" }, { name: 1, price: 1, _id: 0 }).sort({ price: -1 }).limit(5)
```

The mental model is a B-tree over documents rather than a table of rows. A collection is a set of BSON documents; each document is one contiguous byte string with a length prefix, so the server can skip over a document it does not need. A field can hold a scalar, a nested object, or an array, and an index on a field inside an array becomes a multikey index with one index entry per element.

BSON is a binary encoding, not text. Keys are repeated in every document, so a long field name costs bytes on every row and shows up in the working-set size. Types survive the round trip: `Decimal128` stays exact for money, `Date` stores UTC milliseconds, and `ObjectId` carries a timestamp plus a random counter.

A common mistake is treating "flexible schema" as "no schema". Without a validator, one writer can store `price` as a string and another as `Decimal128`, and range queries then silently miss documents because BSON types are compared within a type ordering. Add `$jsonSchema` validation for the invariants you cannot live without, and decide field names deliberately because they are stored in every document.

In [ ]:
from decimal import Decimal

import mongomock

client = mongomock.MongoClient()          # stands in for MongoClient("mongodb://...")
db = client.shop

db.products.insert_many([
    {"sku": "A1", "name": "Espresso Machine", "price": 249.0, "tags": ["kitchen", "coffee"],
     "stock": {"warehouse": "DFW", "qty": 12}},
    {"sku": "B2", "name": "Grinder", "price": 89.0, "tags": ["coffee"]},
    {"sku": "C3", "name": "Kettle", "price": "59.00", "tags": ["kitchen"]},   # wrong type!
])

coffee = list(db.products.find({"tags": "coffee"}, {"_id": 0, "name": 1, "price": 1}).sort("price", -1))
assert [p["name"] for p in coffee] == ["Espresso Machine", "Grinder"]
assert db.products.find_one({"stock.warehouse": "DFW"})["sku"] == "A1"   # dotted path into a subdocument

# Range queries compare within a BSON type: the string price is silently missed.
cheap = [p["sku"] for p in db.products.find({"price": {"$lt": 100}})]
assert cheap == ["B2"], cheap
print("coffee:", coffee, "| price < 100:", cheap, "(C3 stored price as a string)")

## How do you model documents in MongoDB?

MongoDB modeling depends on access patterns. Embed related data when it is usually read together and has bounded size. Reference related data when it grows independently, is shared by many documents, or would create very large documents.

The rule of thumb is one-to-few embeds, one-to-many references, and one-to-squillions (unbounded, such as events or messages) uses a separate collection plus the bucket pattern so a single document never grows without limit. References cost an extra round trip and an application-side join, or a `$lookup` in the pipeline.

```json
{
  "_id": "order_1001",
  "customer_id": "C123",
  "items": [
    {"sku": "A1", "qty": 2},
    {"sku": "B9", "qty": 1}
  ]
}
```

The mechanics are a size and ownership decision, not a style preference. Embedding keeps one document as the aggregate and gives a single atomic read and write. Referencing splits the data into two collections joined by an application-side lookup or `$lookup`, which costs a round trip and cannot be updated atomically unless you use a transaction.

Bound the embed. The 16 MB document limit is hard, and an array that grows without limit also forces the storage engine to rewrite the document as it grows, which moves data and dirties more pages. The bucket pattern groups a fixed number of children per document, for example one document per device per hour, so documents stay a predictable size.

```javascript
// Bucket pattern: one document per sensor per hour, not one per reading
db.readings.insertOne({
  sensor_id: "s-42",
  bucket_start: ISODate("2026-10-04T13:00:00Z"),
  count: 360,
  readings: [ { ts: ISODate("2026-10-04T13:00:01Z"), temp: 21.4 } ]
})

// $lookup is the server-side join; it needs an index on the foreign field
db.orders.aggregate([
  { $match: { _id: "order_1001" } },
  { $lookup: { from: "customers", localField: "customer_id",
               foreignField: "_id", as: "customer" } }
])
```

The mistake to name in an interview is embedding a "one-to-squillions" relationship, such as every event for a user, because the array grows until it hits 16 MB and write latency degrades long before that. Also avoid duplicating a field that changes often: duplication is a cache, so every copy needs an owner and a refresh path, or the two copies drift.

In [ ]:
db = client.modeling
db.customers.insert_one({"_id": "C123", "name": "Ada", "region": "EU"})
db.orders.insert_many([
    {"_id": "order_1001", "customer_id": "C123",                        # reference
     "items": [{"sku": "A1", "qty": 2, "price": 249.0}, {"sku": "B9", "qty": 1, "price": 20.0}]},  # embed
    {"_id": "order_1002", "customer_id": "C123", "items": [{"sku": "B9", "qty": 3, "price": 20.0}]},
])

# Embedded: one read returns the whole aggregate.
order = db.orders.find_one({"_id": "order_1001"})
assert len(order["items"]) == 2

# Referenced: $lookup is the server-side join (index the foreignField in production).
joined = list(db.orders.aggregate([
    {"$match": {"_id": "order_1001"}},
    {"$lookup": {"from": "customers", "localField": "customer_id", "foreignField": "_id", "as": "customer"}},
    {"$unwind": "$customer"},
    {"$project": {"_id": 1, "customer.name": 1, "item_count": {"$size": "$items"}}},
]))
assert joined == [{"_id": "order_1001", "customer": {"name": "Ada"}, "item_count": 2}]
print(joined)

## When should you use MongoDB, and when not?

The engine-level reason is the aggregate: a document is the unit of atomicity and of locality, so a read that touches one document is one page fetch, while the same read in a normalized schema crosses several tables and several pages. A replica set then adds read capacity and automatic failover, and sharding adds write capacity and dataset size. Each property comes from a different component, so be precise about which one you are claiming.

The trade is that you must design around the query patterns you know. A document model optimized for one access path makes a different one awkward, and there is no cost-based join planner to rescue an unplanned `$lookup`. Choose MongoDB when the top queries are aggregate-shaped and the schema moves; choose a relational warehouse when the top queries are ad-hoc joins over many entities.

| Aspect | RDBMS | MongoDB |
|---|---|---|
| Unit | row in a table | BSON document in a collection |
| Schema | fixed, DDL migrations | flexible, optional `$jsonSchema` validation |
| Relationships | foreign keys + joins | embedding, `$lookup`, application joins |
| Transactions | multi-row by default | single-document atomic; multi-document with sessions |
| Scale-out | mostly vertical, read replicas | native sharding + replica sets |
| Query language | SQL | MQL / aggregation pipeline, SQL via Atlas SQL |
| Best fit | strongly relational, reporting | aggregate-oriented, evolving, high write volume |

A quick test: write the three most frequent queries as documents. If most of them touch a single collection and return a document-shaped result, MongoDB fits. If they join four entities and aggregate across them, a relational engine or warehouse fits. Mixed workloads are normal, so many teams keep MongoDB as the operational store and stream changes into a warehouse for analytics.

The hard limits to check before committing are the 16 MB document size, the absence of foreign keys, and the fact that a unique index is per collection, so cross-collection uniqueness needs its own mechanism. None of these are blockers, but each one changes a design decision you would otherwise make implicitly.

## How does MongoDB store data?

Under the hood the default storage engine is WiredTiger: documents are stored in compressed files (snappy by default for collections, zlib/zstd configurable), each collection has a `_id` index, and the journal (write-ahead log) provides durability. BSON supports types JSON does not, such as `ObjectId`, `Decimal128`, `Date`, `Binary`, and 64-bit integers, which matters for money and timestamps.

WiredTiger is the storage engine. Its cache holds uncompressed pages for read and write, and it uses multi-version concurrency control, so readers never block writers. The default cache size is the larger of half of (RAM minus 1 GB) and 256 MB, which is why a server with 64 GB of RAM gets about 31.5 GB of cache and the rest is left for the operating system, connections, and the aggregation pipeline. The cache evicts clean pages easily and dirty pages only after they are written; when the dirty fraction crosses about 20 percent, application threads are recruited to help evict and latency rises.

Durability comes from two mechanisms that are often confused. The journal is a write-ahead log on disk; with `j: true` the write is acknowledged only after it reaches the journal, and the journal is flushed about every 100 ms by default. A checkpoint is a consistent snapshot of the data files, taken about every 60 seconds or after 2 GB of journal, and recovery replays the journal from the last checkpoint.

```javascript
// Cache counters: read these before blaming a query
const wt = db.serverStatus().wiredTiger.cache
print(wt["bytes currently in the cache"])
print(wt["maximum bytes configured"])
print(wt["pages read into cache"])
print(wt["tracked dirty bytes in the cache"])

// Checkpoint and transaction counters
db.serverStatus().wiredTiger.transaction["transaction checkpoint most recent time (msecs)"]

// Override the cache size at startup
// mongod --wiredTigerCacheSizeGB 8
```

## How do indexes affect MongoDB query performance?

Mechanically a MongoDB index is a B-tree keyed by the indexed fields plus the record id, stored as its own WiredTiger file. An `IXSCAN` walks that tree to a range of keys, then a `FETCH` stage reads the matching documents. A `COLLSCAN` reads every document in the collection. The planner picks the winning plan by racing candidate plans and keeping the one with the best score.

A compound index is usable when the query has an equality prefix on the leading fields. The order of the fields is the order of the sort inside the B-tree, so `{ customer_id: 1, created_at: -1 }` supports "one customer, newest first" but not a sort by `created_at` alone. The sort direction can be reversed by walking the tree backwards, but mixed directions inside one index only support that exact direction or its full reverse.

Index types, and the one-line reason each exists. A single-field index supports equality and sort on one field. A compound index supports a query on several fields and the ESR ordering. A multikey index is a compound or single index whose field holds an array; it stores one key per element, cannot cover a query, and cannot index two array fields in the same compound index. A text index supports `$text` search with stemming and weights, and a collection may have only one. A TTL index deletes documents after a date, runs a background monitor about every 60 seconds, and must be a single-field index on a date or an array of dates. A partial index indexes only documents matching `partialFilterExpression`, which makes it smaller and cheaper. A wildcard index (`$**`) indexes arbitrary or unknown field names, which is flexible but larger and slower than a targeted index. A unique index enforces uniqueness, and a sparse index (legacy) skips documents missing the field.

```javascript
db.users.createIndex({ email: 1 }, { unique: true })
db.orders.createIndex({ customer_id: 1, created_at: -1 })          // compound
db.products.createIndex({ tags: 1 })                                // multikey (tags is an array)
db.products.createIndex({ name: "text", description: "text" },
                        { weights: { name: 10, description: 2 } })
db.sessions.createIndex({ expires_at: 1 }, { expireAfterSeconds: 0 })  // TTL
db.orders.createIndex({ tenant_id: 1 }, { partialFilterExpression: { status: "open" } })
db.logs.createIndex({ "$**": 1 })                                   // wildcard
db.stores.createIndex({ location: "2dsphere" })                     // geospatial
db.events.createIndex({ ts: 1 }, { expireAfterSeconds: 7776000 })   // delete after 90 days
```

In [ ]:
db = client.indexing
orders = db.orders
orders.create_index([("customer_id", 1), ("created_at", -1)], name="customer_recent")
orders.create_index("expires_at", expireAfterSeconds=0)
db.users.create_index("email", unique=True, partialFilterExpression={"email": {"$exists": True}})

db.users.insert_many([{"name": "no-email-1"}, {"name": "no-email-2"}])   # allowed: partial index
db.users.insert_one({"email": "ada@example.com"})
try:
    db.users.insert_one({"email": "ada@example.com"})
    raise AssertionError("unique index should reject the duplicate")
except mongomock.DuplicateKeyError:
    print("duplicate email rejected by the unique index")

names = sorted(orders.index_information())
assert names == ["_id_", "customer_recent", "expires_at_1"], names
print("orders indexes:", names)

### How do you design indexes with the ESR rule and covered queries?

Write the query in terms of roles before writing the index. Equality predicates pin exact key prefixes. The sort field determines the order the B-tree must deliver. Range predicates scan a contiguous span. ESR orders them Equality, then Sort, then Range. Equality fields can be in any relative order, though putting the most selective first can help when the planner chooses a prefix. The sort field must follow in the requested direction so the blocking `SORT` stage disappears. A range field placed last still narrows the scan, but any field after a range cannot be used efficiently.

Covered queries are the other lever. If the index contains every field the query filters on and projects, the server answers from the index alone: no `FETCH` stage and `totalDocsExamined` is 0. Exclude `_id` in the projection unless it is part of the index. A multikey index cannot cover a query on the array field, and an index cannot cover a query that needs a document field that is not in the index.

```javascript
// Query roles: tenant_id and status are equality, created_at is the sort, total is a range
db.orders.createIndex({ tenant_id: 1, status: 1, created_at: -1, total: 1 },
                      { name: "esr_orders" })

// Covered: index contains filter and projection fields, _id excluded
db.orders.find(
  { tenant_id: "t1", status: "open" },
  { _id: 0, tenant_id: 1, status: 1, created_at: 1 }
).sort({ created_at: -1 }).explain("executionStats")

// Verify which index the planner chose, and force one when testing
db.orders.find({ tenant_id: "t1" }).hint("esr_orders").explain("executionStats")

// Find unused indexes before dropping anything
db.orders.aggregate([{ $indexStats: {} }])
```

The common mistake is ordering fields by perceived selectivity, for example putting a range date before an equality status because the date is "more selective". That forces an in-memory sort and may scan more keys. The other is expecting a covered query while projecting `_id`; `_id` is fetched unless the index includes it or the projection excludes it. A third is creating several near-duplicate compound indexes; each one adds write cost and the planner may pick the wrong one.

### How do you read explain("executionStats")?

Tuning starts by making the plan visible. `explain()` has three modes: `queryPlanner` (the winning plan and the rejected candidates, no execution), `executionStats` (runs the winning plan and reports work counters), and `allPlansExecution` (runs the candidates too, for comparing). Use `queryPlanner` to see which index won, and `executionStats` to see how much work the winner did. In a sharded cluster, run `explain` on `mongos` so you see the merge stage and each shard's plan.

Read three counters together. `nReturned` is how many documents the query returned. `totalKeysExamined` is how many index keys were read. `totalDocsExamined` is how many full documents were fetched. For a good query, `totalDocsExamined` equals `nReturned` and `totalKeysExamined` is close to `nReturned`. A large `totalKeysExamined / nReturned` ratio means the index is not selective enough for the filter; a large `totalDocsExamined / nReturned` ratio means the index found many candidates that the filter then rejected.

```json
{
  "queryPlanner": {
    "winningPlan": {
      "stage": "FETCH",
      "inputStage": { "stage": "IXSCAN", "indexName": "customer_id_1_created_at_-1" }
    }
  },
  "executionStats": {
    "nReturned": 3,
    "executionTimeMillis": 0,
    "totalKeysExamined": 3,
    "totalDocsExamined": 3,
    "executionStages": {
      "stage": "FETCH",
      "nReturned": 3,
      "docsExamined": 3,
      "inputStage": { "stage": "IXSCAN", "keysExamined": 3, "nReturned": 3 }
    }
  }
}
```

The stage names to recognize: `COLLSCAN` means no index was used; `IXSCAN` is an index range scan; `FETCH` reads the documents the index pointed at; `SORT` is a blocking in-memory sort; `PROJECTION_COVERED` or an `IXSCAN` with no `FETCH` parent means the index covered the query; `SHARD_MERGE` and `SINGLE_SHARD` are the router stages. A `SORT` stage with `usedDisk: true` means the sort spilled, which `allowDiskUse` permits and `memLimit` limits.

A `COLLSCAN` is not automatically wrong: a query that returns most of a small collection is cheaper as a scan. The ratio decides. On a large collection, `totalDocsExamined` in the millions with a small `nReturned` is the signal to add or reorder an index.

## How does MongoDB scale?

Vertical scaling is one bigger machine and has a hard ceiling plus a single failure domain. Horizontal scaling is more machines, and MongoDB's horizontal story has two layers that are easy to conflate. A replica set copies the entire dataset to each member, so it scales reads and availability only. A sharded cluster splits each collection into chunks on a shard key, so it scales writes and dataset size. If a single collection is too large or too hot, only sharding helps.

Before distributing, shrink the problem. The cheapest levers in order are: fix the index so the query stops scanning, project only the fields you need, move cold data out with a TTL or Atlas online archive, and precompute rollups with `$merge` so dashboards read a small collection. Sharding adds a router hop, a shard key you cannot easily change, and operational surface, so it should be the last lever, not the first.

### What is in a sharded cluster?

- [shard](https://www.mongodb.com/docs/manual/core/sharded-cluster-shards/#std-label-shards-concepts): Each shard contains a subset of the sharded data. Each shard must be deployed as a [replica set.](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-replica-set)
- [mongos](https://www.mongodb.com/docs/manual/core/sharded-cluster-query-router/): The `mongos` acts as a query router, providing an interface between client applications and the sharded cluster. [`mongos`](https://www.mongodb.com/docs/manual/reference/program/mongos/#mongodb-binary-bin.mongos) can support [hedged reads](https://www.mongodb.com/docs/manual/core/sharded-cluster-query-router/#std-label-mongos-hedged-reads) to minimize latencies.
- [config servers](https://www.mongodb.com/docs/manual/core/sharded-cluster-config-servers/#std-label-sharding-config-server): Config servers store metadata and configuration settings for the cluster. As of MongoDB 3.4, config servers must be deployed as a replica set (CSRS).

![Diagram of a sample sharded cluster for production purposes.  Contains exactly 3 config servers, 2 or more ``mongos`` query routers, and at least 2 shards. The shards are replica sets.](https://www.mongodb.com/docs/manual/images/sharded-cluster-production-architecture.bakedsvg.svg)

Mechanically, a sharded collection is divided into chunks, each a contiguous range of shard key values with a default maximum size of 128 MB. The `mongos` routers keep a cached copy of the chunk-to-shard map from the config servers, and the balancer moves chunks between shards when counts diverge. A query is targeted when it includes the shard key or a prefix of a compound shard key; otherwise `mongos` broadcasts to every shard and merges results, which multiplies latency by the slowest shard.

Shard keys cannot be changed on a collection before MongoDB 5.0; 5.0 adds `reshardCollection`, and 4.4 adds refining a shard key by adding a suffix field. This is why the key is a design decision made at creation time.

```javascript
sh.enableSharding("shop")
sh.shardCollection("shop.orders", { tenant_id: 1, order_id: 1 })

sh.status()                 // chunk distribution per collection
sh.getBalancerState()       // is the balancer running?
db.chunks.find({ ns: "shop.orders" }).count()   // chunk count
```

```json
{"queryPlanner": {"winningPlan": {"stage": "SHARD_MERGE", "shards": [
  {"shardName": "shardA", "winningPlan": {"stage": "IXSCAN"}},
  {"shardName": "shardB", "winningPlan": {"stage": "COLLSCAN"}}
]}}}
```

The cost is paid on every query that does not carry the shard key. Scatter-gather sends the query to every shard, so latency becomes the slowest shard and throughput is bounded by the slowest shard, not by total capacity. Target operations by including the shard key prefix in the query and in the leading fields of an index on each shard.

### Hashed vs ranged sharding: how do you choose a shard key?

A shard key is judged on cardinality, frequency, and monotonicity. Cardinality: enough distinct values that chunks can split and spread. Frequency: no single value dominates, or one chunk holds most documents and cannot split below one value per chunk. Monotonicity: values that increase over time send every insert to the highest chunk, so one shard takes all writes. The key must also appear in the queries you care about, or those queries become scatter-gather.

Hashed sharding fixes monotonicity by hashing the value, at the cost of range locality. Ranged compound keys keep locality: route on a high-cardinality field and include a date for range queries, for example `{ tenant_id: 1, created_at: 1 }`. Chunks are about 128 MB by default, and the balancer migrates chunks when counts diverge; zone rules pin ranges to shards for data residency.

Hashed sharding hashes the shard key value and stores chunks over the hash space, which spreads monotonically increasing keys evenly but destroys range locality. Ranged sharding keeps the natural order, so a range query on the shard key is targeted, but a monotonic key concentrates writes. A compound key is often the right compromise: a high-cardinality routing field first for even spread, then a date second for locality, for example `{ tenant_id: 1, created_at: 1 }`.

Zones let you pin ranges of the shard key to specific shards, which is how you keep European data in European data centers while still sharding globally. Since MongoDB 4.4 you can refine a shard key by appending a suffix field without resharding; since 5.0 `reshardCollection` can change the key entirely, but it is an online, expensive operation.

```javascript
// Hashed: even writes, but range queries broadcast
sh.shardCollection("shop.events", { device_id: "hashed" })

// Ranged compound: targeted by tenant, local by date
sh.shardCollection("shop.orders", { tenant_id: 1, created_at: 1 })

// Zone: keep EU tenants on EU shards
sh.addShardToZone("shardEU", "EU")
sh.updateZoneKeyRange("shop.orders", { tenant_id: "eu-a" }, { tenant_id: "eu-z" }, "EU")

// Refine a shard key by appending a field (4.4+)
db.adminCommand({ refineCollectionShardKey: "shop.orders",
                  key: { tenant_id: 1, created_at: 1, order_id: 1 } })
```

The hot-shard problem is easy to see in a simulation: route 10,000 monotonically increasing `order_id`s to four shards by range and by hash.

In [ ]:
import hashlib
from collections import Counter

SHARDS = 4
existing_max = 1_000_000                  # chunks were split when ids were below this
new_ids = range(existing_max, existing_max + 10_000)


def ranged_shard(order_id: int) -> int:
    # Four equal ranges over the ids seen at split time; everything new is past the last bound.
    return min(order_id * SHARDS // existing_max, SHARDS - 1)


def hashed_shard(order_id: int) -> int:
    digest = hashlib.md5(str(order_id).encode()).digest()
    return int.from_bytes(digest[:8], "big") % SHARDS


ranged = Counter(ranged_shard(i) for i in new_ids)
hashed = Counter(hashed_shard(i) for i in new_ids)
assert ranged == {3: 10_000}                       # every insert hits the last chunk
assert all(2_200 < hashed[s] < 2_800 for s in range(SHARDS))
print("ranged inserts per shard:", dict(ranged))
print("hashed inserts per shard:", dict(sorted(hashed.items())))

## How do atomicity and transactions work in MongoDB?

In MongoDB, a write operation is [atomic](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-atomic-operation) on the level of a single document, even if the operation modifies multiple embedded documents *within* a single document.

When a single write operation (e.g. [`db.collection.updateMany()`](https://www.mongodb.com/docs/manual/reference/method/db.collection.updateMany/#mongodb-method-db.collection.updateMany)) modifies multiple documents, the modification of each document is atomic, but the operation as a whole is not atomic.

A single-document write is atomic because it is one BSON document mutated under a WiredTiger transaction; either the whole update applies or none of it does. Multi-document transactions use the same storage-engine transaction machinery extended across documents and shards with snapshot isolation: the transaction reads a consistent snapshot at a point in time and writes are invisible until commit. They require a replica set or sharded cluster, default to a 60-second lifetime, and are limited by a 16 MB total oplog entry for the transaction.

```javascript
// Atomic on one document: no transaction needed
db.accounts.updateOne(
  { _id: "acct-1", balance: { $gte: 100 } },     // guard in the filter
  { $inc: { balance: -100 }, $push: { ledger: { amount: -100, at: new Date() } } }
)

// Multi-document: explicit session and transaction
const session = db.getMongo().startSession()
session.startTransaction({ readConcern: { level: "snapshot" }, writeConcern: { w: "majority" } })
try {
  session.getDatabase("bank").accounts.updateOne({ _id: "A" }, { $inc: { balance: -100 } })
  session.getDatabase("bank").accounts.updateOne({ _id: "B" }, { $inc: { balance: 100 } })
  session.commitTransaction()
} catch (e) {
  session.abortTransaction()
  throw e
}
```

In [ ]:
db = client.bank
db.accounts.insert_one({"_id": "acct-1", "balance": 150, "ledger": []})


def withdraw(account: str, amount: int) -> bool:
    result = db.accounts.update_one(
        {"_id": account, "balance": {"$gte": amount}},             # guard in the filter
        {"$inc": {"balance": -amount}, "$push": {"ledger": {"amount": -amount}}},
    )
    return result.modified_count == 1


assert withdraw("acct-1", 100) is True
assert withdraw("acct-1", 100) is False                         # would overdraw: no match
account = db.accounts.find_one({"_id": "acct-1"})
assert account["balance"] == 50 and account["ledger"] == [{"amount": -100}]
print(account)

## What is an aggregation pipeline in MongoDB?

Stages come in two shapes. Streaming stages process one document at a time and pass it on: `$match`, `$project`, `$addFields`, `$unset`, `$limit`, `$skip`, and `$unwind`. Blocking stages must consume all input before they emit anything: `$group`, `$sort`, `$bucket`, `$facet`, and `$count`. A blocking stage means the pipeline cannot return a first result until the whole input is seen, and it holds memory or spills to disk.

Ordering is therefore a performance decision. Push `$match` and `$project` immediately after the source so fewer documents and fewer fields reach the blocking stages. A `$sort` placed before a `$match` sorts documents that will be thrown away. A `$match` before `$sort` lets the planner use an index for the sort and avoid the blocking SORT stage entirely.

```text
db.orders.aggregate([
  { $match: { status: "paid", created_at: { $gte: ISODate("2026-10-01") } } },
  { $unwind: "$items" },
  { $group: {
      _id: { day: { $dateTrunc: { date: "$created_at", unit: "day" } }, sku: "$items.sku" },
      revenue: { $sum: { $multiply: ["$items.qty", "$items.price"] } },
      orders: { $sum: 1 },
      avg_qty: { $avg: "$items.qty" }
  } },
  { $sort: { "_id.day": 1, revenue: -1 } },
  { $merge: { into: "daily_sku_revenue", whenMatched: "replace", whenNotMatched: "insert" } }
], { allowDiskUse: true })
```

Put `$match` and `$limit` first so indexes can be used and fewer documents flow through the pipeline. Stage names to know: `$match`, `$project`, `$group`, `$sort`, `$unwind`, `$lookup`, `$facet` (several reports in one pass), `$bucket`, `$setWindowFields` (running totals), and `$merge`/`$out` for materialized views. `$lookup` does not use an index on the foreign collection's join field unless one exists, so index `foreignField`.

`$lookup` is the expensive stage. In the common `localField`/`foreignField` form the server probes the foreign collection once per input document, and if `foreignField` is not indexed that probe is a collection scan, turning an N-document pipeline into N scans. Always index the foreign field. The `let`/`pipeline` form runs a sub-pipeline per input document and is more flexible but not cheaper.

`$facet` runs several sub-pipelines over the same input in one pass, which is efficient for building several report sections at once, but each sub-pipeline shares the 100 MB memory limit and cannot contain `$out` or `$merge`. `allowDiskUse: true` lets blocking stages spill to disk past the memory limit, which prevents failure at the cost of disk I/O.

In [ ]:
db = client.analytics
db.orders.insert_many([
    {"tenant_id": "t1", "status": "paid", "items": [{"sku": "A1", "qty": 2, "price": 10.0},
                                                   {"sku": "B2", "qty": 1, "price": 5.0}]},
    {"tenant_id": "t1", "status": "paid", "items": [{"sku": "A1", "qty": 1, "price": 10.0}]},
    {"tenant_id": "t1", "status": "open", "items": [{"sku": "B2", "qty": 9, "price": 5.0}]},
    {"tenant_id": "t2", "status": "paid", "items": [{"sku": "C3", "qty": 1, "price": 99.0}]},
])

pipeline = [
    {"$match": {"tenant_id": "t1", "status": "paid"}},          # first: can use an index
    {"$project": {"_id": 0, "items": 1}},
    {"$unwind": "$items"},
    {"$group": {"_id": "$items.sku",
                "revenue": {"$sum": {"$multiply": ["$items.qty", "$items.price"]}},
                "lines": {"$sum": 1}}},
    {"$sort": {"revenue": -1}},
    {"$limit": 10},
]
top = list(db.orders.aggregate(pipeline))
assert top == [{"_id": "A1", "revenue": 30.0, "lines": 2}, {"_id": "B2", "revenue": 5.0, "lines": 1}]

report = list(db.orders.aggregate([
    {"$match": {"tenant_id": "t1"}},
    {"$facet": {
        "by_status": [{"$group": {"_id": "$status", "orders": {"$sum": 1}}}, {"$sort": {"_id": 1}}],
        "total": [{"$count": "orders"}],
    }},
]))[0]
assert report["by_status"] == [{"_id": "open", "orders": 1}, {"_id": "paid", "orders": 2}]
assert report["total"] == [{"orders": 3}]
print("top SKUs:", top)
print("facet report:", report)

```javascript
// $lookup is cheap only when the foreign field is indexed
db.customers.createIndex({ _id: 1 })                       // already there
db.customers.createIndex({ region: 1 })                    // for the let/pipeline form

db.orders.aggregate([
  { $match: { created_at: { $gte: ISODate("2026-10-01") } } },
  { $lookup: { from: "customers", localField: "customer_id",
               foreignField: "_id", as: "customer" } },
  { $unwind: "$customer" },
  { $facet: {
      byRegion: [{ $group: { _id: "$customer.region", orders: { $sum: 1 } } }],
      topCustomers: [{ $group: { _id: "$customer_id", spend: { $sum: 1 } } },
                     { $sort: { spend: -1 } }, { $limit: 10 }]
  } }
], { allowDiskUse: true })
```

## How do write concern, read concern, and read preference work?

Write concern has two independent parts: how many members must acknowledge, and whether the acknowledgement waits for the journal. `w: 0` is fire-and-forget. `w: 1` acknowledges from the primary's memory. `w: "majority"` acknowledges once a majority of voting members have applied the write, so it survives a failover. Adding `j: true` makes the acknowledgement wait for the on-disk journal. `wtimeout` bounds the wait so a slow secondary does not block forever.

Read concern decides what data a read is allowed to see. `local` returns whatever the node has, which can be rolled back. `majority` returns data acknowledged by a majority and therefore not rolled back. `snapshot` returns a consistent point-in-time view and is what transactions use. `linearizable` reads the latest majority-acknowledged write and waits; it is the strictest and the slowest. Read preference chooses which member serves the read: `primary` is consistent, `secondaryPreferred` trades freshness for read scale.

```text
// Durable, majority-acknowledged, journaled write
db.orders.insertOne(
  { _id: "order_1003", total: NumberDecimal("99.00") },
  { writeConcern: { w: "majority", j: true, wtimeout: 5000 } }
)
```

The common mistake is assuming `w: "majority"` also means durable to disk; without `j: true` the journal may be up to about 100 ms behind, so a power loss can still lose acknowledged writes. The other mistake is reading from secondaries with default `local` read concern and then reasoning about the result as if it were current. Secondary reads are eventually consistent, and replication lag is a real, monitorable number.

```python
from pymongo import MongoClient
from pymongo.read_concern import ReadConcern
from pymongo.read_preferences import ReadPreference
from pymongo.write_concern import WriteConcern

orders = MongoClient("mongodb://mongo.internal:27017/?replicaSet=rs0").shop.orders

# Durable write: acknowledged by a majority and journaled
orders.with_options(write_concern=WriteConcern(w="majority", j=True, wtimeout=5000)).insert_one(
    {"_id": "order_2001", "total": 99}
)
# Read that will not observe rolled-back data
orders.with_options(read_concern=ReadConcern("majority"), read_preference=ReadPreference.PRIMARY).find_one(
    {"_id": "order_2001"}
)
# Scale reads to secondaries, accepting eventual consistency
orders.with_options(read_preference=ReadPreference.SECONDARY_PREFERRED).find({"status": "paid"}).limit(5)
```

### How does MongoDB fit into the CAP theorem?

Map the guarantees to settings. A partition happens regardless of configuration, so the real choice is what the majority side does and what the minority side does. The majority side can elect a primary and continue accepting writes; the minority side cannot elect one, so its members become read-only. That is the CP behavior you get by default with `w: "majority"` and `readConcern: "majority"`. Choosing `w: 1` and reading from secondaries moves you toward AP: writes succeed on the majority side sooner, minority-side secondaries can still answer reads, but a failover can roll back recent writes and those reads can be stale.

The common mistake is calling MongoDB simply "CP". It is tunable per operation, and the deployment's behavior depends on the concerns you set, not on a fixed label. Say instead: with `w: "majority"` and `readConcern: "majority"` it behaves as CP; with `w: 1` and secondary reads it trades consistency for availability. Also note that losing the majority makes the set unavailable for writes no matter how you tune the client, because an election cannot complete.

## What is a replica set and how does it work?

Replication works through the oplog, a capped collection in the `local` database. The primary records every write as an idempotent oplog entry; secondaries tail the oplog and apply entries in order, recording their position. A secondary's `optime` is how far it has applied, and replication lag is the difference between the primary's latest optime and a secondary's. Idempotency means an entry can be replayed safely after a restart, which is what makes the oplog the single mechanism behind replication, change streams, and retryable writes.

Failover is an election, not a promotion. Members exchange heartbeats; if a primary stops responding for `electionTimeoutMillis` (default 10 seconds), a member that is caught up and has a majority of votes calls an election. A majority of voting members must be reachable to elect a primary, which is why an even number of voters is worse than an odd number.

```javascript
rs.status().members.forEach(m => print(m.name, m.stateStr, m.optimeDate, m.health))
rs.printReplicationInfo()        // oplog size and time window
rs.printSecondaryReplicationInfo()

// Oplog window: how far behind a consumer may fall
db.getSiblingDB("local").oplog.rs.stats().maxSize
db.getSiblingDB("local").oplog.rs.find().sort({ $natural: -1 }).limit(1)

rs.stepDown(60)
```

Write concern and the journal interact with replication. `w: "majority"` waits for a majority of members to apply the write, which bounds failover data loss to writes below the commit point. `j: true` guarantees the primary wrote to its journal, which protects against a primary crash but not against losing the primary before replication. For the strongest combination, use `w: "majority"` with `j: true` and `readConcern: "majority"`.

The common mistake is running an even number of voting members or adding arbiters to break a tie. Two arbiters in a three-member set can elect a primary without any data-bearing member being current. Use three or five data-bearing voters, and use arbiters only to reach an odd voter count when a data-bearing member is not possible.

## How do change streams work in MongoDB?

Change streams are built on the oplog. The server opens a cursor over a filtered view of oplog entries, converts them to a stable event format, and returns a `resumeToken` with each event. Because the token is the oplog position, a consumer can resume exactly after the last processed event with `resumeAfter`, or after a timestamp with `startAtOperationTime`. Change streams require a replica set or sharded cluster, and their availability depends on the oplog retention window.

The event shape is stable across versions: `operationType` (`insert`, `update`, `replace`, `delete`, `invalidate`, and DDL types), `ns`, `documentKey`, `fullDocument` for insert and replace, and `updateDescription` for update. For an update you only get the changed fields unless you pass `fullDocument: "updateLookup"`, which fetches the current document. On a sharded cluster, each shard produces events and `mongos` merges them in cluster time order.

```javascript
const stream = db.orders.watch(
  [{ $match: { "fullDocument.status": "paid",
               operationType: { $in: ["insert", "update"] } } }],
  { fullDocument: "updateLookup", maxAwaitTimeMS: 1000 }
)

while (!stream.isClosed()) {
  if (stream.hasNext()) {
    const change = stream.next()
    print(change.operationType, change.documentKey._id, change.clusterTime)
    // change._id is the resume token for this event
  }
}
```

```python
# pymongo consumer that resumes after a crash (requires a replica set)
resume_token = load_token()                       # persisted after the last downstream commit
pipeline = [{"$match": {"operationType": {"$in": ["insert", "update"]}}}]
with orders.watch(pipeline, resume_after=resume_token, full_document="updateLookup") as stream:
    for change in stream:
        publish_to_kafka(change["fullDocument"])  # or write to a search index
        save_token(stream.resume_token)
```

The common mistake is treating a change stream as durable queue storage. If the consumer stays offline longer than the oplog window, the resume token is no longer in the oplog and the stream fails with `ChangeStreamHistoryLost`; the consumer must resynchronize from a full read. Size the oplog for your worst acceptable downtime, monitor the oplog window, and persist the resume token only after the downstream write commits, so a crash reprocesses rather than skips. Make consumers idempotent for the same reason.

## How does MongoDB handle concurrency?

WiredTiger uses document-level concurrency with MVCC. Writers take a write conflict check on the document they mutate; readers see a snapshot and are never blocked. Two transactions writing the same document produce a `WriteConflict` that the driver retries. A slow operation that matches many documents still touches many documents, so long scans that write are the main source of conflicts.

The two application-side tools are optimistic concurrency with a version field, and idempotent retries. Never hold a transaction open across a user think-time or a network call to another service; keep the transaction to the database work and retry on transient errors.

```text
// Optimistic concurrency: version field guards against lost updates
const res = db.documents.updateOne(
  { _id: "doc1", version: 42 },
  { $set: { body: "new" }, $inc: { version: 1 } }
)
if (res.matchedCount === 0) {
  throw new Error("conflict: document changed since it was read")
}
```

In [ ]:
db = client.cms
db.documents.insert_one({"_id": "doc1", "body": "v1", "version": 1})


def save(doc_id: str, read_version: int, body: str) -> bool:
    result = db.documents.update_one(
        {"_id": doc_id, "version": read_version},
        {"$set": {"body": body}, "$inc": {"version": 1}},
    )
    return result.matched_count == 1


snapshot = db.documents.find_one({"_id": "doc1"})          # two editors read version 1
assert save("doc1", snapshot["version"], "edit by Ada") is True
assert save("doc1", snapshot["version"], "edit by Bob") is False   # lost update prevented
assert db.documents.find_one({"_id": "doc1"}) == {"_id": "doc1", "body": "edit by Ada", "version": 2}
print("second writer got a conflict and must re-read")

## How do you manage schema design, changes, and integrity?

Schema change is a data migration, not a DDL statement, because there is no central catalog to alter. Adding a field means writing it, so you run a batched backfill that updates a bounded number of documents per round to avoid holding locks and to spread I/O. Removing a field is `$unset`, which can also be batched. Changing a type is the risky one: readers must handle both shapes while the backfill runs.

The safe sequence is expand, migrate, contract. Expand: add the new field and write both old and new shapes. Migrate: backfill in batches, verify counts, and only then switch readers. Contract: stop writing the old field and remove it later. Version each document with `schema_version` so code can branch during the window.

```text
// JSON Schema validation: enforce shape, types, and required fields
db.createCollection("customers", {
  validator: {
    $jsonSchema: {
      bsonType: "object",
      required: ["email", "tier", "created_at"],
      additionalProperties: false,
      properties: {
        email: { bsonType: "string", pattern: "^.+@.+$" },
        tier: { enum: ["free", "pro", "enterprise"] },
        created_at: { bsonType: "date" },
        schema_version: { bsonType: "int", minimum: 1 }
      }
    }
  },
  validationLevel: "moderate",     // existing invalid docs are allowed
  validationAction: "error"        // or "warn"
})
```

The batched backfill, runnable: bound each round by `_id`, never update the whole collection in one statement.

In [ ]:
db = client.migrations
db.customers.insert_many([{"_id": i, "email": f"user{i}@example.com"} for i in range(2_500)])


def backfill(batch_size: int = 1_000) -> list[int]:
    rounds, last = [], None
    while True:
        query = {"schema_version": {"$exists": False}}
        if last is not None:
            query["_id"] = {"$gt": last}
        ids = [d["_id"] for d in db.customers.find(query, {"_id": 1}).sort("_id", 1).limit(batch_size)]
        if not ids:
            return rounds
        db.customers.update_many({"_id": {"$in": ids}}, {"$set": {"schema_version": 2}})
        last = ids[-1]
        rounds.append(len(ids))


rounds = backfill()
assert rounds == [1_000, 1_000, 500]
assert db.customers.count_documents({"schema_version": 2}) == 2_500
print("rounds:", rounds)

Integrity in MongoDB is enforced per document by default. A unique index enforces uniqueness and doubles as the guard for idempotent inserts. `$jsonSchema` enforces shape, types, and required fields at write time. A single-document update keeps an invariant atomic. Anything spanning documents needs a transaction, and anything spanning collections needs either a transaction or an accepted orphan risk plus reconciliation.

Referential integrity is therefore a choice. Pay on write with a transaction, or pay later with a periodic job that finds and repairs orphans. A common pattern is to store a reference and a denormalized copy of the fields you read, then treat the copy as a cache with a defined refresh owner and an audit query that detects drift.

The common mistake is assuming a unique index on a field that can be missing works as expected. In MongoDB, a missing field is indexed as `null`, so a unique index rejects the second document that also lacks the field unless the index is partial or sparse. Use a partial index such as `{ partialFilterExpression: { email: { $exists: true } } }` when only present values must be unique.

The best practices reduce to three questions asked per collection. What is the aggregate that must be read or written atomically? How bounded is it? Which queries must be indexed, and in what order? Embed when the child is bounded and owned by the parent; reference when it is shared, unbounded, or large. Keep nesting shallow so dotted field names stay readable and updates stay targeted.

Then make the invariants explicit: `$jsonSchema` for shape and required fields, unique and partial indexes for uniqueness, and a version field for migrations. Choose types deliberately, `Decimal128` for money and `Date` for time, because comparability depends on it.

## How do you migrate from an RDBMS to MongoDB?

Migration mechanics follow the access patterns. First list the top queries; then decide the aggregate boundaries. Usually one row becomes one document, but parent and child tables often merge into one embedded document, and many-to-many join tables become arrays of references or are embedded on one side. Preserve types: `NUMERIC` and `DECIMAL` become `Decimal128`, timestamps become `Date`, and relational ids stay as `_id` so references remain stable.

Cutover is the hard part. Either dual-write from the application, or capture changes with CDC and apply them to MongoDB, then verify by comparing counts, checksums, and spot checks on joined data. Keep the relational system as the source of truth until the comparison is clean, then switch, and keep the old system readable for a rollback window.

In [ ]:
import sqlite3

sql = sqlite3.connect(":memory:")
sql.executescript("""
CREATE TABLE orders (order_id TEXT PRIMARY KEY, customer_id TEXT, created_at TEXT);
CREATE TABLE order_items (order_id TEXT, sku TEXT, qty INTEGER, price TEXT);
INSERT INTO orders VALUES ('o1', 'C1', '2026-10-01'), ('o2', 'C2', '2026-10-02');
INSERT INTO order_items VALUES ('o1', 'A1', 2, '249.00'), ('o1', 'B9', 1, '20.00'), ('o2', 'B9', 3, '20.00');
""")

docs = {}
for order_id, customer_id, created_at, sku, qty, price in sql.execute("""
    SELECT o.order_id, o.customer_id, o.created_at, i.sku, i.qty, i.price
    FROM orders o JOIN order_items i USING (order_id) ORDER BY o.order_id
"""):
    doc = docs.setdefault(order_id, {"_id": order_id, "customer_id": customer_id,   # keep relational id
                                     "created_at": created_at, "items": []})
    doc["items"].append({"sku": sku, "qty": qty, "price": str(Decimal(price))})  # Decimal128 in pymongo

target = client.migrated.orders
target.insert_many(list(docs.values()), ordered=False)
target.create_index([("customer_id", 1), ("created_at", -1)])

# Verify: counts on both sides before cutover.
source_lines = sql.execute("SELECT COUNT(*) FROM order_items").fetchone()[0]
target_lines = sum(len(d["items"]) for d in target.find())
assert target.count_documents({}) == 2 and source_lines == target_lines == 3
print(target.find_one({"_id": "o1"}))

## How do you handle large data volumes?

Follow an order that stops as early as possible. Reduce first: TTL indexes delete expired documents, and `$merge` rollups replace raw events with small summaries. Split second: the bucket pattern groups many readings into one document, and time series collections store measurements in compressed buckets with a `metaField` for the series identity and an optional `expireAfterSeconds` for retention. Distribute third: shard on a key with high cardinality and even writes, ideally the same field that scopes the common query. Archive last: Atlas online archive moves cold documents to object storage while keeping them queryable.

Time series collections are the modern answer for measurements. They bucket by time automatically, compress well, and support the same query language, so the bucket pattern is usually only needed when you must update inside the bucket.

```text
// Time series collection with automatic expiry
db.createCollection("sensor_readings", {
  timeseries: {
    timeField: "ts",
    metaField: "sensor",
    granularity: "minutes"
  },
  expireAfterSeconds: 7776000      // 90 days
})
```

## How do you secure MongoDB?

Security is layered and each layer is enforced by a different component. Authentication proves identity (SCRAM is the default, x.509 and LDAP for enterprise). Authorization is role-based: a role is a named set of privileges of the form `{ resource, actions }`, and users receive roles. Encryption in transit is TLS between clients and between replica set members; encryption at rest is file-level by default and can be client-side field level (CSFLE) for specific fields. Auditing records administrative and authentication events. Network controls are the outermost layer: bind to private interfaces, allow-list addresses, and require TLS between members.

CSFLE has a queryability consequence. Deterministic encryption produces the same ciphertext for the same plaintext, so equality queries still work; randomized encryption is stronger but the field becomes unqueryable, which means you must store a separate lookup token if you need to search it.

```text
// Least-privilege application user with a custom role
db.createRole({
  role: "ordersApp",
  privileges: [
    { resource: { db: "shop", collection: "orders" }, actions: ["find", "insert", "update"] }
  ],
  roles: []
})
db.createUser({
  user: "orders-svc",
  pwd: passwordPrompt(),
  roles: [{ role: "ordersApp", db: "shop" }]
})
```

## What are the backup and restore options?

- mongodump and mongorestore: Command-line tools for creating binary exports of database contents and restoring them.
- Filesystem snapshots: For creating point-in-time snapshots of data (only consistent if taken with the journal, or on a stopped/fsync-locked node).
- MongoDB Atlas Backup: Continuous backups and point-in-time recovery for cloud-hosted databases.
- MongoDB Ops Manager: For on-premises deployments, offering backup automation and point-in-time recovery.

```bash
# Logical backup of one database
mongodump --uri="mongodb://localhost:27017" --db=shop --out=/backup/2026-10-04

# Point-in-time-consistent backup of a replica set (captures oplog for replay)
mongodump --uri="mongodb://localhost:27017/?replicaSet=rs0" --oplog --out=/backup/rs-2026-10-04

# Restore into a different database name
mongorestore --uri="mongodb://localhost:27017" --nsFrom="shop.*" --nsTo="shop_restore.*" /backup/2026-10-04
```

The options differ by whether they capture a consistent point in time. A logical dump (`mongodump`) walks collections and can drift unless it runs with `--oplog`, which records oplog entries written during the dump so the restore can replay to a consistent point. A filesystem snapshot is only consistent if it includes the journal or the node was stopped or fsync-locked. Atlas continuous backup and Ops Manager capture the oplog continuously, which is what gives point-in-time recovery.

The common mistake is treating a backup as proven before restoring it. Test the restore into an isolated environment on a schedule, and check that indexes, validators, and users are recreated, since `mongodump` captures them but `mongoexport` does not. Also remember that `mongoexport` writes text and loses BSON types such as `Decimal128` and `Date` precision, so it is an interchange format, not a backup.

## How do you monitor and optimize MongoDB performance?

Monitoring has two layers. Server-level metrics tell you about resources: `db.serverStatus()` exposes connections, opcounters, and the WiredTiger cache; `mongostat` samples the counters once per interval; `mongotop` shows per-collection read and write time. Operation-level metrics tell you about queries: `db.currentOp()` shows what is running now, the slow query log records anything over `slowms`, and `$indexStats` shows which indexes are actually used.

Tuning follows the numbers, not a hunch. Read `explain("executionStats")` before and after a change, and compare the same counters. Watch the WiredTiger cache dirty fraction and page eviction rate: if the working set does not fit in cache, latency rises even when the plan reads the index correctly. Check replication lag and the oplog window, because a lagging secondary constrains `w: "majority"` latency.

```text
// Find the slowest operations currently in flight
db.adminCommand({ currentOp: true, secs_running: { $gt: 1 } })
```

```bash
# Server-level counters and per-collection activity
mongostat --uri="mongodb://localhost:27017" --rowcount=5
mongotop 5
```

## How does MongoDB handle geospatial queries?

A `2dsphere` index indexes GeoJSON points and shapes on a sphere and supports `$near`, `$nearSphere`, `$geoWithin`, and `$geoIntersects`. A legacy `2d` index works on flat coordinate pairs and is rarely what you want. The index itself is a B-tree over a space-filling curve of the geometry, so a proximity query walks nearby cells. The `$geoNear` aggregation stage must be the first stage of the pipeline and requires a geospatial index.

Coordinates are always `[longitude, latitude]` in GeoJSON, in that order. A polygon must be a closed ring with the first and last positions equal, and a `2dsphere` query uses meters for radius in `$maxDistance` when the distance is in `$geoNear` with a spherical index.

```text
db.stores.createIndex({ location: "2dsphere" })
```

The common mistake is swapping longitude and latitude, which puts the point in the wrong hemisphere and silently returns nothing. The other is using `$near` in an aggregation pipeline: `$near` is a query operator and is not allowed there, so use `$geoNear` as the first stage instead.

## What is MongoDB Atlas?

Atlas is managed MongoDB: it provisions replica sets and sharded clusters, runs backups, exposes the Performance Advisor and index suggestions, and adds services that are not in the server, such as Atlas Search (Lucene), Atlas Vector Search, Atlas Stream Processing, and online archive that moves cold documents to object storage while keeping them queryable through the same connection.

The interview-relevant part is what Atlas does not change. The document model, indexes, shard keys, and read and write concerns behave the same, so a tuning problem on Atlas is still an `explain` problem. Atlas changes operations, not the query engine.

## What are the trade-offs and limitations?

- Consistency vs availability: `w: "majority"` plus `readConcern: "majority"` is durable and rollback-safe, but each write waits for a majority round trip and the cluster refuses writes if it loses the majority. `w: 1` with secondary reads is faster and more available, but a failover can roll writes back and replicas can be stale.
- Write amplification: every secondary index must be updated on every write, and a multikey index is updated once per array element. A collection with 12 indexes turns one insert into 13 index writes; on write-heavy workloads, indexes are often the bottleneck, not the disk.
- Hot partitions and hot shards: a monotonically increasing shard key (timestamp, ObjectId) concentrates writes in the last chunk; a low-cardinality key (status, country) creates chunks that cannot split. Hashed sharding spreads writes but makes range queries scatter-gather.
- Document growth and the 16 MB limit: unbounded `$push` arrays cause rewrites and eventually hard failures. Cap arrays with `$slice`, or move children into their own collection.
- Memory pressure: index and working-set size must fit in RAM. Once the WiredTiger cache is exhausted, `IXSCAN` still reads from disk and latency jumps even though the plan looks correct.
- Weak analytics: no cost-based join planner, no columnar execution. If the workload is mostly arbitrary ad-hoc joins and aggregations across many entities, a warehouse or a relational database is the better home for it.
- When NOT to use MongoDB: strictly relational, constraint-heavy transactional systems (banking cores, double-entry ledgers needing foreign keys and multi-table ACID by default); heavy OLAP/BI with star-schema joins; workloads requiring cross-collection uniqueness or complex cascading deletes; teams without the operational maturity to run replica sets and sharding; and any case where a small relational database with a handful of tables would simply work.

## MongoDB exercises

These exercises model the tuning arithmetic with the fields MongoDB reports: index efficiency is `nReturned / totalKeysExamined`, WiredTiger's default cache is the larger of half of (RAM minus 1 GB) and 256 MB, and TTL retention is expressed in seconds.

### Try it 1. Order compound index fields by the ESR rule

**Task.** Given query fields tagged as equality, sort, or range, build the index key order that follows the ESR rule.

**Hint.** Equality fields first, then the sort field with its direction, then range fields.

**Expected result.** `[('status', 1), ('tenant_id', 1), ('created_at', -1), ('total', 1)]`.

In [ ]:
FIELDS = [
    {"name": "created_at", "role": "sort", "direction": -1},
    {"name": "status", "role": "equality"},
    {"name": "total", "role": "range"},
    {"name": "tenant_id", "role": "equality"},
]


def esr_index(fields):
    ordered = [f for f in fields if f["role"] == "equality"]
    ordered += [f for f in fields if f["role"] == "sort"]
    ordered += [f for f in fields if f["role"] == "range"]
    return [(f["name"], f.get("direction", 1)) for f in ordered]


key = esr_index(FIELDS)
print("index key:", key)

assert key == [("status", 1), ("tenant_id", 1), ("created_at", -1), ("total", 1)]
assert key[-1][0] == "total", "range fields must come last"

### Try it 2. Compute index selectivity from nReturned and totalKeysExamined

**Task.** Compute index efficiency as `nReturned / totalKeysExamined` for a poorly selective plan and for a good one, and classify each.

**Hint.** Efficiency near 1 means almost every key read produced a returned document; below about 0.5 means the index is examined too broadly.

**Expected result.** The poor plan has efficiency `0.00025` and verdict `reorder the index`; the good plan has efficiency `0.962` and verdict `index is selective`.

In [ ]:
def index_efficiency(n_returned, total_keys_examined):
    return n_returned / total_keys_examined


def verdict(efficiency):
    return "index is selective" if efficiency >= 0.5 else "reorder the index"


poor = index_efficiency(25, 100000)
good = index_efficiency(50, 52)
print(f"poor  plan: nReturned=25  totalKeysExamined=100000 -> {poor:.5f} ({verdict(poor)})")
print(f"good  plan: nReturned=50  totalKeysExamined=52     -> {good:.3f} ({verdict(good)})")

assert round(poor, 5) == 0.00025
assert round(good, 3) == 0.962
assert verdict(poor) == "reorder the index"
assert verdict(good) == "index is selective"

### Try it 3. Detect a COLLSCAN from an explain summary

**Task.** Given an execution summary with the winning stage and the work counters, classify the plan.

**Hint.** `COLLSCAN` or a zero `totalKeysExamined` means no index was used. Otherwise compare `totalDocsExamined` and `totalKeysExamined` with `nReturned`.

**Expected result.** The summary classifies as `COLLSCAN: add an index on the filter fields`.

In [ ]:
def diagnose(plan):
    if plan["stage"] == "COLLSCAN" or plan["totalKeysExamined"] == 0:
        return "COLLSCAN: add an index on the filter fields"
    if plan["totalDocsExamined"] > 10 * plan["nReturned"]:
        return "IXSCAN is not selective: reorder the index"
    if plan["totalKeysExamined"] > 10 * plan["nReturned"]:
        return "index too coarse: add an equality field"
    return "plan is healthy"


summary = {
    "stage": "COLLSCAN",
    "nReturned": 12,
    "totalKeysExamined": 0,
    "totalDocsExamined": 1240000,
}

result = diagnose(summary)
print("stage:", summary["stage"])
print("docs examined per returned:", summary["totalDocsExamined"] // summary["nReturned"])
print("verdict:", result)

assert result == "COLLSCAN: add an index on the filter fields"
assert diagnose({"stage": "IXSCAN", "nReturned": 50, "totalKeysExamined": 52,
                 "totalDocsExamined": 50}) == "plan is healthy"

### Try it 4. Choose a shard key from cardinality and write distribution

**Task.** Score candidate shard keys by cardinality and write spread, reject candidates that are not in the queries or have too few distinct values, and pick the best.

**Hint.** A usable key must appear in the queries and have at least a thousand distinct values; among usable keys, prefer high cardinality and even write spread, and penalize monotonic keys by their low write spread.

**Expected result.** `tenant_id` wins with a score near `2.97`; `status` is rejected.

In [ ]:
import math

CANDIDATES = [
    {"name": "created_at", "cardinality": 1_000_000, "write_spread": 0.10, "in_queries": True},
    {"name": "tenant_id", "cardinality": 2_000, "write_spread": 0.90, "in_queries": True},
    {"name": "status", "cardinality": 5, "write_spread": 0.95, "in_queries": False},
]


def score(candidate):
    if not candidate["in_queries"] or candidate["cardinality"] < 1000:
        return -1.0
    return math.log10(candidate["cardinality"]) * candidate["write_spread"]


ranked = sorted(CANDIDATES, key=score, reverse=True)
for c in ranked:
    print(f"{c['name']:12s} score={score(c):.3f}")
print("chosen shard key:", ranked[0]["name"])

assert ranked[0]["name"] == "tenant_id"
assert round(score(CANDIDATES[1]), 2) == 2.97
assert score(CANDIDATES[2]) == -1.0, "status is not in the queries"

### Try it 5. Estimate the WiredTiger cache from RAM

**Task.** Compute the default WiredTiger cache size for hosts with 64 GB, 8 GB, and 2 GB of RAM.

**Hint.** The default is the larger of half of (RAM minus 1 GB) and 256 MB; 256 MB is 0.25 GB.

**Expected result.** `31.5` GB, `3.5` GB, and `0.5` GB (the 256 MB floor for the 2 GB host after the subtraction).

In [ ]:
def wiredtiger_cache_gb(ram_gb):
    return max(0.5 * (ram_gb - 1), 0.25)


for ram in (64, 8, 2):
    cache = wiredtiger_cache_gb(ram)
    print(f"RAM {ram:>2} GB -> cache {cache:>5.2f} GB")

assert wiredtiger_cache_gb(64) == 31.5
assert wiredtiger_cache_gb(8) == 3.5
assert wiredtiger_cache_gb(2) == 0.5

### Try it 6. Size a TTL index from retention days

**Task.** Convert a 90-day retention policy into the `expireAfterSeconds` value and estimate how many readings the index must track for a fixed ingest rate.

**Hint.** Seconds per day is 86400. Documents retained is `per_second * 86400 * days`; the TTL monitor deletes on a background pass, so a document can outlive its expiry by up to about 60 seconds.

**Expected result.** `expireAfterSeconds` is `7776000`, about `31.1` billion readings are retained, and the monitor delay bound is `60` seconds.

In [ ]:
def ttl_seconds(days):
    return days * 86400


retention_days = 90
expire_after = ttl_seconds(retention_days)
readings_per_second = 4000
retained = readings_per_second * 86400 * retention_days
ttl_monitor_delay_seconds = 60

print("expireAfterSeconds:", expire_after)
print("retained readings: %.1f billion" % (retained / 1e9))
print("TTL monitor delay bound:", ttl_monitor_delay_seconds, "seconds")

assert expire_after == 7776000
assert round(retained / 1e9, 1) == 31.1
assert ttl_monitor_delay_seconds == 60

### Try it 7. Model the cost of $lookup from collection sizes

**Task.** Given an orders collection and a customers collection, compare the index probes an indexed `$lookup` performs with the document reads an unindexed one performs.

**Hint.** With an index on `foreignField` the server probes once per input document; without one it scans the foreign collection once per input document.

**Expected result.** The indexed form does about `2,000,000` reads (one probe plus one fetch per order) and the unindexed form `500,000,000,000`; the unindexed form is `250000` times more work.

In [ ]:
orders = 1_000_000
customers = 500_000

indexed_reads = orders * 2          # one index probe plus one foreign document fetch per order
unindexed_reads = orders * customers  # a full scan of customers per order
slowdown = unindexed_reads / indexed_reads

print("orders:", orders, "customers:", customers)
print("indexed lookups  :", indexed_reads)
print("unindexed lookups:", unindexed_reads)
print("slowdown factor  :", int(slowdown))

assert indexed_reads == 2_000_000
assert unindexed_reads == 500_000_000_000
assert slowdown == 250000.0

### Try it 8. Triage a plan by documents examined per returned document

**Task.** Compute `totalDocsExamined / nReturned` for an execution summary and decide whether the index is earning its keep.

**Hint.** A healthy index scan examines one document per returned document; a ratio far above 1 means the index found many candidates that the filter rejected.

**Expected result.** The ratio is `100.0` and the verdict is `poor selectivity: examine fewer documents per result`.

In [ ]:
def triage(n_returned, total_docs_examined):
    ratio = total_docs_examined / n_returned
    if ratio <= 1.5:
        return ratio, "healthy: about one document per result"
    return ratio, "poor selectivity: examine fewer documents per result"


ratio, verdict = triage(n_returned=12, total_docs_examined=1200)
print(f"docsExamined/nReturned = {ratio:.1f}")
print("verdict:", verdict)

assert ratio == 100.0
assert verdict == "poor selectivity: examine fewer documents per result"
assert triage(50, 50)[0] == 1.0

## Real world use cases for MongoDB

### 1. E-commerce product catalog

Business context: a marketplace with 8 million SKUs, each category having different attributes (clothes have sizes and colors, electronics have voltage and warranty). Volume: ~8M products, ~25M variants, read-heavy at roughly 50:1 reads to writes, 3,000 reads/s at peak. Access patterns: fetch one product page with all variants and images, search by category and tags, update price and stock frequently.

Schema design: one document per product embedding variants and a bounded image list; references to a `categories` collection so category metadata is not duplicated 8M times. A `2dsphere` index is not needed; a compound index on `{ category_id: 1, updated_at: -1 }` serves category browsing.

```text
db.products.insertOne({
  _id: "sku-1001",
  name: "Espresso Machine",
  category_id: "cat-kitchen",
  attributes: { voltage: "110V", warranty_months: 24 },
  tags: ["coffee", "kitchen"],
  variants: [
    { sku: "sku-1001-black", color: "black", price: NumberDecimal("249.00"), stock: 12 },
    { sku: "sku-1001-steel", color: "steel", price: NumberDecimal("269.00"), stock: 3 }
  ],
  updated_at: new Date()
})
```

Why this design: the product page is a single-document read, and `variants` is bounded (a few dozen at most), so embedding wins. The heterogeneous `attributes` object is exactly the flexible-schema case a relational catalog needs EAV tables for.

### 2. IoT time series with automatic expiry

Business context: 40,000 sensors reporting every 10 seconds (about 345M readings/day, tens of billions over the retention window). Access patterns: dashboards read one sensor for the last 24 hours, rarely query across all sensors; older than 90 days is discarded.

Schema design: a time series collection with `metaField` on the sensor and a TTL, or the bucket pattern (one document per sensor per hour) when you need sub-document updates.

```text
db.createCollection("sensor_readings", {
  timeseries: { timeField: "ts", metaField: "sensor", granularity: "seconds" },
  expireAfterSeconds: 7776000
})
```

Why this design: time series collections store measurements columnar-ish by bucket internally, compress well, and TTL removes expired data without a delete job. Sharding uses `{ sensor.id: "hashed" }` or a compound `{ "sensor.id": 1, ts: 1 }` so writes spread across shards while a single-sensor range query stays targeted.

### 3. User activity feed

Business context: a social app with 50M users; the timeline of a user with 5,000 followers must load in under 100 ms. Access patterns: read the newest 50 items for one user, append new items at high rate, keep only the last few hundred items.

Schema design: fan-out on write into a per-user feed document that keeps a bounded array using `$push` with `$slice`; if a user's feed can exceed a few thousand items, use a dedicated `feed_items` collection keyed by `{ user_id, created_at }`.

In [ ]:
db = client.social


def push_to_timeline(user: str, item: dict, keep: int = 500) -> None:
    db.timelines.update_one(
        {"_id": user},
        {"$push": {"items": {"$each": [item], "$position": 0, "$slice": keep}}},   # bounded array
        upsert=True,
    )


for n in range(510):
    push_to_timeline("user-42", {"actor": f"user-{n % 7}", "verb": "liked", "seq": n})
timeline = db.timelines.find_one({"_id": "user-42"})["items"]
assert len(timeline) == 500 and timeline[0]["seq"] == 509     # newest first, capped at 500
print("timeline length:", len(timeline), "| newest:", timeline[0])

Why this design: reads dominate, so precomputing the feed on write (`$push` with `$slice`) removes the read-time fan-out entirely. For celebrities whose feeds would exceed 16 MB, the separate collection with a compound index keeps documents bounded and lets a shard key of `{ user_id: 1 }` localize each feed.

### 4. Fintech ledger and idempotent payments

Business context: a payments API writing double-entry ledger entries and transferring funds between accounts. Volume: ~5M transactions/day, strict correctness, auditable history. Access patterns: look up a transaction by idempotency key, list the last 100 entries for an account, move money between two accounts atomically.

Schema design: `accounts` and `ledger_entries` collections; a unique index on the idempotency key so a retried request cannot double-post; a multi-document transaction for the transfer; `w: "majority"` everywhere.

The idempotency half runs on `mongomock`; the cross-document transfer needs `client.start_session()` + `session.start_transaction(write_concern={"w": "majority"})` on a real replica set (see the transactions section).

In [ ]:
db = client.ledger
db.payments.create_index("idempotency_key", unique=True)
db.accounts.insert_many([{"_id": "acct-1", "balance": 10_000}, {"_id": "acct-2", "balance": 0}])


def post_payment(key: str, source: str, target: str, amount: int) -> str:
    try:
        db.payments.insert_one({"idempotency_key": key, "status": "posted", "amount": amount})
    except mongomock.DuplicateKeyError:
        return "duplicate"                                   # retry is a no-op
    # In production these writes share the payment insert's transaction.
    db.accounts.update_one({"_id": source}, {"$inc": {"balance": -amount}})
    db.accounts.update_one({"_id": target}, {"$inc": {"balance": amount}})
    db.ledger_entries.insert_many([{"account_id": source, "amount": -amount, "ref": key},
                                   {"account_id": target, "amount": amount, "ref": key}])
    return "posted"


assert post_payment("req-7f3a", "acct-1", "acct-2", 5_000) == "posted"
assert post_payment("req-7f3a", "acct-1", "acct-2", 5_000) == "duplicate"
balances = {a["_id"]: a["balance"] for a in db.accounts.find()}
assert balances == {"acct-1": 5_000, "acct-2": 5_000}
assert sum(e["amount"] for e in db.ledger_entries.find()) == 0  # double entry balances
print(balances)

Why this design: `$inc` is atomic and the unique idempotency index makes retries safe; the transaction spans collections only because the accounting invariant demands it. Replicas give durability, and change streams feed the downstream audit and reporting systems.

### 5. Multi-tenant SaaS

Business context: a B2B platform with 2,000 tenants and wildly different data volumes (one tenant is 30% of all rows). Volume: 900M documents, retention differs per tenant. Access patterns: every query is scoped to one tenant; schema varies slightly per tenant plan; older data expires after the contract ends.

Schema design: a shared collection with `tenant_id` in every document and in the leading position of every index; per-tenant `$jsonSchema` variants are avoided in favor of a superset validator plus optional fields; TTL or archival per tenant.

```text
// Every index leads with tenant_id so reads are always targeted
db.tickets.createIndex({ tenant_id: 1, status: 1, created_at: -1 })
```

Why this design: a shared collection with a tenant prefix is the cheapest multi-tenant model and makes isolation a matter of discipline plus a mandatory index prefix. Truly regulated or enormous tenants can be moved to their own database and cluster without changing the application's tenant-scoped access pattern.